# Két lemez közös éllel

A két síklemez közös Gmsh-élhez csatlakozik. A középfelület elmozdulása és a fizikai forgás két háromkomponensű globális mező. A helyi alakváltozásokat lemezenként vetítjük ki.

In [12]:
using LowLevelFEM, LinearAlgebra#, SparseArrays

openGeometry("folded-plate.geo")

In [13]:
mat = Material("plates", E=2e5, ν=0.3)

U = Field([mat], type=:VectorField, dim=3, field=:u)
Φ = Field([mat], type=:VectorField, dim=3, field=:φ, reducedOrder=false);

Egy síklemezen az irányok állandók, ezért mindegyik mezőből elég az első elemi csomópont értéke.

In [14]:
n = normalVector(U, "plates")
tξ, tη = tangentVectors(U, "plates");

In [15]:
n = [n[1], n[2], n[3]]
tξ = [tξ[1], tξ[2], tξ[3]]
tη = [tη[1], tη[2], tη[3]];

In [16]:
# kron(tξ, tη)' ⋅ Grad(U) gives tξᵀ(∇U)tη.

A_u = vcat(kron(tξ,tξ)',
           kron(tη,tη)',
           (kron(tξ,tη) + kron(tη,tξ))',
           zeros(3,9),
           kron(n,tη)',
           kron(n,tξ)')

A_φ = vcat(zeros(3,9),
           kron(tη,tξ)',
           -kron(tξ,tη)',
           (kron(tη,tη) - kron(tξ,tξ))',
           zeros(2,9))

G_φ = vcat(zeros(6,3), -tξ', tη')

B = A_u ⋅ Grad(U) + A_φ ⋅ Grad(Φ) + G_φ ⋅ Φ;

In [17]:
b = 0.12
k_s = 4/5

C = D(:PlaneStress, mat)

D1 = b*C

D2 = b^3 / 12 * C

D3 = k_s * b * mat.μ * Matrix{Float64}(I, 2, 2)

D_plate = cat(D1, D2, D3; dims=(1,2))

K = ∫(B' ⋅ D_plate ⋅ B; Γ="plates")
K[:,:]

1386×1386 SparseArrays.SparseMatrixCSC{Float64, Int64} with 43449 stored entries:
⎡⡿⣯⣙⢿⡓⢚⡷⡿⠲⠰⠆⣇⣍⡁⣋⣉⠀⠀⠂⠒⣿⣯⡓⢿⡓⣻⢾⡇⠶⠰⢸⣎⣍⡑⣃⡁⠀⠐⠐⠒⎤
⎢⣷⣜⠻⣦⡀⠈⡟⢣⣾⠘⠘⠛⠛⣃⣈⢉⢀⡀⠀⠀⣧⣙⢿⣦⡀⢩⡞⣧⡆⠚⠘⠙⢛⣁⡈⣁⢀⠀⠀⠀⎥
⎢⣹⢈⡀⠈⢿⣷⡀⠀⠀⠀⠀⠀⠀⣩⡏⣈⢨⢡⣷⣷⡇⢉⡀⠘⢿⣶⠀⠀⠀⠀⠀⠀⠈⣽⡇⡉⣌⣼⢷⢷⎥
⎢⣽⡯⠿⣉⠀⠈⣻⢞⣛⡿⠷⠶⠤⠍⠀⠉⠈⠀⠀⠀⣽⠮⢿⣉⠀⢙⡿⣟⣻⠿⠶⠶⠤⠁⠀⠁⠀⠀⠀⠀⎥
⎢⢘⡂⣚⠛⠀⠀⣿⡼⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⢓⡂⣛⠃⠀⠸⣿⠻⣿⣷⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠬⢥⣶⠀⠀⠀⢹⡇⠈⠻⣿⣿⣦⠀⠀⠀⠀⠀⠀⠀⠬⣥⠶⠀⠀⠀⢻⡇⠙⢻⣻⣿⣄⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠇⠹⠿⢠⡄⣠⡄⠇⠀⠀⠈⠛⠿⣧⣤⣤⣤⣄⣀⡀⠏⠿⠏⢠⡄⣤⠘⠇⠀⠀⠉⠻⢻⣤⣤⣤⣤⣄⣀⠀⎥
⎢⡏⢸⡆⢘⡋⢩⡄⠀⠀⠀⠀⠀⠀⣿⣻⣾⣯⡉⠉⠉⡇⢿⡄⢘⡋⣭⠀⠀⠀⠀⠀⠀⢸⣯⣳⣝⣯⡉⠉⠉⎥
⎢⠀⠀⠀⠰⠆⣒⠂⠀⠀⠀⠀⠀⠀⢿⡏⠻⣿⣿⣦⡀⠀⠀⠀⠰⠆⣒⠀⠀⠀⠀⠀⠀⠈⣿⡌⢿⣿⣿⣦⠀⎥
⎢⢨⠀⠀⠀⢽⣿⠀⠀⠀⠀⠀⠀⠀⠸⡇⠀⠈⠻⣿⣿⡅⠀⠀⠨⣿⣶⠀⠀⠀⠀⠀⠀⠀⢸⡇⠀⠈⠻⣿⣷⎥
⎢⡿⣿⣍⢻⡍⢉⡳⡟⠹⠰⠆⣧⣯⡅⣭⣍⠀⠀⠁⠉⣿⣿⡉⢻⡍⣙⢿⡇⠷⠰⢸⣮⣯⡍⣥⡅⠀⠈⠈⠉⎥
⎢⣽⣌⠻⣷⣀⠈⡟⢳⠿⠘⠘⠃⠋⣁⣀⢉⢀⡀⡀⡀⣧⣈⠿⣧⣀⢩⠟⡷⠇⠛⠘⠘⢋⣁⡀⣁⢀⢀⡀⡀⎥
⎢⣽⣨⡄⣈⢻⣷⣄⢀⣀⡀⠀⠀⠀⣭⡏⣬⢨⢡⢻⣿⣇⢩⡄⣘⢻⣶⡀⣀⣀⠀⠀⠀⠈⣽⡇⡍⣬⣼⢳⢷⎥
⎢⠾⠷⠾⣭⠀⠀⣿⢯⣿⡛⠿⠶⠶⠄⠀⠀⠀⠀⠀⠀⠿⠷⢿⡥⠀⢨⡿⣯⣿⡻⠿⠶⠦⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢘⡃⣨⠉⠀⠀⣿⡞⢿⣿⣷⣀⠀⠀⠀⠀⠀⠀⠀⠀⢙⡃⣭⠁⠀⠘⣿⡻⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡲⢶⣖⠀⠀⠀⢸⡇⠀⠹⣿⣾⣧⡀⠀⠀⠀⠀⠀⠀⡲⣶⣒⠀⠀⠀⢻⡇⠈⠻⣿⣿⣦⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢇⠹⠟⢰⣆⣤⠄⠃⠀⠀⠀⠙⠛⣶⡶⣶⣦⣤⣀⣀⡏⠿⠏⢰⣆⣤⠈⠃⠀⠀⠈⠛⠻⣦⡶⣶⣦⣤⣀⡀⎥
⎢⠍⠸⠆⢨⡍⠩⠄⠀⠀⠀⠀⠀⠀⣿⣝⢾⣦⣍⠉⠉⠅⠿⠄⢨⡍⠭⠀⠀⠀⠀⠀⠀⢸⣯⣿⣿⣮⡉⠉⠉⎥
⎢⢀⠀⠀⠐⣂⣽⠀⠀⠀⠀⠀⠀⠀⢿⡏⠻⣿⣿⣦⡀⡀⠀⠀⢐⣂⣿⠀⠀⠀⠀⠀⠀⠈⣿⡎⠻⣿⣿⣦⡀⎥
⎣⢰⠀⠀⠀⢽⣗⠀⠀⠀⠀⠀⠀⠀⠘⡇⠀⠈⠛⢿⣿⡆⠀⠀⠨⢽⣖⠀⠀⠀⠀⠀⠀⠀⠸⡇⠀⠈⠻⢿⣷⎦

In [18]:
#A_d = reshape(0.5 * (kron(tξ,tη) - kron(tη,tξ)), 1, 9)
A_d = (kron(tξ,tη) - kron(tη,tξ))' / 2
#B_d = A_d ⋅ Grad(U) + reshape(n, 1, 3) ⋅ Φ
B_d = A_d ⋅ Grad(U) + n' ⋅ Φ

α = 1e-4
Kd = ∫(B_d' ⋅ [α * b * mat.μ;;] ⋅ B_d; Γ="plates")
Kd[:,:]

1386×1386 SparseArrays.SparseMatrixCSC{Float64, Int64} with 30128 stored entries:
⎡⡿⣯⣙⢿⡓⢚⡷⡿⠲⠰⠆⣇⣍⡁⣋⣉⠀⠀⠂⠒⣿⣯⡓⢿⠒⣻⠶⡿⠲⠰⢸⣎⣍⡑⣃⡁⠀⠐⠐⠒⎤
⎢⣷⣜⠻⣦⡀⠈⡟⢣⡾⠘⠘⠛⠛⣃⣈⢉⢀⡀⠀⠀⣧⣙⢿⣦⡀⢩⡟⣧⡇⠛⠘⠙⢛⣁⡉⣁⢀⠀⠀⠀⎥
⎢⣹⢈⡀⠈⢿⣷⡀⠀⠀⠀⠀⠀⠀⣩⡏⣈⢨⢡⣷⣷⡇⢉⠀⠸⢿⣦⠀⠀⠀⠀⠀⠀⠈⣽⡇⡉⣌⣼⢶⢷⎥
⎢⣽⡯⠿⣉⠀⠈⣻⢞⣛⡿⠷⠶⠤⠍⠀⠉⠈⠀⠀⠀⣽⡯⢿⣉⠀⢙⡿⣟⣛⠿⠷⠶⠤⠁⠀⠁⠉⠀⠀⠀⎥
⎢⢘⡂⣚⠋⠀⠀⣿⡼⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⢓⡂⣛⠃⠀⠰⣿⡺⣿⣷⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠬⢥⣶⠀⠀⠀⢹⡇⠈⠻⣿⣿⣦⠀⠀⠀⠀⠀⠀⠀⠬⣥⣶⠀⠀⠀⢻⡇⠈⢻⣷⣽⣆⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠇⠹⠿⢠⡄⣠⡄⠇⠀⠀⠈⠛⠿⣧⣤⣤⣤⣄⣀⡀⠏⠿⠏⢠⡄⣤⠈⠇⠀⠀⠈⠻⢻⣤⣤⣤⣤⣄⣀⠀⎥
⎢⡏⢸⡆⢘⡋⢩⡄⠀⠀⠀⠀⠀⠀⣿⣻⣾⣯⡉⠉⠉⡇⢿⠀⢘⡋⣭⠀⠀⠀⠀⠀⠀⢸⣏⡳⣿⣏⡉⠉⠉⎥
⎢⠀⠀⠀⠰⠆⣒⠂⠀⠀⠀⠀⠀⠀⢿⡏⠻⣿⣿⣦⡀⠀⠀⠀⠰⠆⣒⠀⠀⠀⠀⠀⠀⠈⣿⡝⢾⣿⣿⣄⠀⎥
⎢⢨⠀⠀⠀⢽⣿⠀⠀⠀⠀⠀⠀⠀⠸⡇⠀⠈⠻⣿⣿⡅⠀⠀⠨⣿⡿⠀⠀⠀⠀⠀⠀⠀⢹⡇⠀⠙⠿⣷⣷⎥
⎢⡿⣿⣍⢻⡍⢉⡷⡿⠹⠰⠆⣧⣯⡅⣭⣍⠀⠀⠁⠉⣿⣿⡉⢻⠉⣹⠷⡟⠱⠰⢸⣎⣯⡍⣥⡅⠀⠈⠈⠉⎥
⎢⣽⣌⠻⣷⣀⡀⡟⢳⠿⠘⠘⠛⠋⣁⣀⢀⢀⡀⡀⡀⣧⣈⠿⣧⣀⢠⠟⣷⠇⠛⠘⠘⢋⣀⡀⣀⢀⢀⠀⡀⎥
⎢⣼⣠⡄⣈⠻⣷⣄⢀⢀⡀⠀⠀⠀⣭⡏⣬⢨⢡⣿⡿⣇⣠⠀⣘⠻⣦⡀⣀⢀⠀⠀⠀⠈⣽⡇⡍⣬⣼⢳⠳⎥
⎢⣼⡧⠿⣭⠀⠀⣿⢯⣻⡻⠿⠶⠦⠄⠀⠀⠀⠀⠀⠀⣽⠧⢿⣥⠀⢨⡿⣯⣻⡻⠿⠶⠦⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢘⡂⣭⠉⠀⠀⣿⡜⢿⣿⣦⣀⠀⠀⠀⠀⠀⠀⠀⠀⢑⡂⣭⠁⠀⠐⣿⡺⣵⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡲⢶⣖⠀⠀⠀⢹⡇⠈⠻⣝⣿⣦⡀⠀⠀⠀⠀⠀⠀⡲⢶⣒⠀⠀⠀⢻⡇⠈⠻⣿⣿⣦⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢇⠹⠟⢰⣆⣤⠄⠃⠀⠀⠈⠙⠛⣶⡶⢶⣦⣤⣄⣀⡏⠿⠋⢰⣆⣤⠈⠃⠀⠀⠈⠛⠻⣦⡶⣶⣦⣤⣀⡀⎥
⎢⠍⠸⠇⢨⡍⠩⠄⠀⠀⠀⠀⠀⠀⣿⣽⣮⣳⣍⠉⠉⠅⠿⠀⢨⡍⠭⠀⠀⠀⠀⠀⠀⢸⣯⣿⣿⣦⡉⠉⠉⎥
⎢⢀⠀⠀⠐⣂⣽⠃⠀⠀⠀⠀⠀⠀⢿⡏⠹⣿⣿⣷⡄⡀⠀⠀⢐⣂⣿⠀⠀⠀⠀⠀⠀⠈⣿⡌⠻⣿⣿⣦⡀⎥
⎣⢰⠀⠀⠀⢼⣗⠀⠀⠀⠀⠀⠀⠀⠘⡇⠀⠀⠙⢽⣿⡆⠀⠀⠠⢽⡒⠀⠀⠀⠀⠀⠀⠀⠸⡇⠀⠈⠻⢿⣷⎦

In [19]:
fu = ∫(U ⋅ [0.0, 0.0, 1.0]; Γ="load")
fφ = ∫(Φ ⋅ [0.0, 1.0, 0.0]; Γ="load")
f = SystemVector([fu, fφ]);

bc_u = BoundaryCondition("clamp"; field=U, ux=0, uy=0, uz=0)
bc_φ = BoundaryCondition("clamp"; field=Φ, φx=0, φy=0, φz=0)

BoundaryCondition("clamp", Problem("folded-plate", :VectorField, 3, 3, Material[Material("plates", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 231, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :rhs, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing)), Dict{Symbol, Union{Function, Number, ScalarField}}(:φz => 0, :φy => 0, :φx => 0))

In [20]:
u, φ = solveField(K + Kd, f, support=[bc_u, bc_φ])

(VectorField(Matrix{Float64}[], [0.0; 0.0; … ; -7.174042530245801e-6; -0.005383003099519284;;], [0.0], Int64[], 1, :v3D, Problem("folded-plate", :VectorField, 3, 3, Material[Material("plates", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 231, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing)), :field), VectorField(Matrix{Float64}[], [0.0; 0.0; … ; 0.04683358933013371; 0.003630405708289085;;], [0.0], Int64[], 1, :v3D, Problem("folded-plate", :VectorField, 3, 3, Material[Material("plates", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 231, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :rhs, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing)), :field)

In [21]:
showDoFResults(u, name="u", visible=true, factor=10)

0

In [22]:
openPostProcessor()